# Location of the Marajó Basin study area

Regional geographic location map. This notebook preserves the visual style of the original Altamira location map, but uses geographic limits appropriate for the Marajó Basin. No sedimentary-basin shapefile is plotted here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

In [ ]:
from matplotlib.lines import Line2D
from pathlib import Path
import zipfile
import geopandas as gpd
import unicodedata
import shutil
import geobr

# 1) Read the basin shapefile from Bacias-zip.zip
zip_path = Path("Bacias-zip.zip")
extract_dir = Path("bacias_extracted")
extract_dir.mkdir(exist_ok=True)

if not zip_path.exists():
    raise FileNotFoundError(
        f"File not found: {zip_path.resolve()}\n"
        "Place Bacias-zip.zip in the same folder in which this notebook is executed."
    )

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

shp_files = sorted(extract_dir.glob("*.shp"))
if not shp_files:
    raise FileNotFoundError("No .shp file was found inside Bacias-zip.zip.")

shp_path = shp_files[0]
bacias = gpd.read_file(shp_path)

# Ensure geographic coordinates
if bacias.crs is None:
    bacias = bacias.set_crs(epsg=4326)
else:
    bacias = bacias.to_crs(epsg=4326)

print(f"Shapefile loaded: {shp_path}")
print(f"Number of sedimentary basins: {len(bacias)}")
print(f"Columns: {list(bacias.columns)}")

# 2) Select only the Marajó Basin
def normalize_text(value):
    value = "" if value is None else str(value)
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return value.strip().casefold()

if "name" not in bacias.columns:
    raise KeyError(
        f"Expected column 'name' was not found. Available columns: {list(bacias.columns)}"
    )

mask_marajo = bacias["name"].map(normalize_text).eq("marajo")
marajo_basin = bacias.loc[mask_marajo].copy()

if marajo_basin.empty:
    raise ValueError("The Marajó Basin was not found in the shapefile.")

# Merge multiple records if necessary, while preserving a single basin geometry
if len(marajo_basin) > 1:
    attrs = marajo_basin.iloc[[0]].drop(columns="geometry").copy()
    geom = marajo_basin.geometry.union_all()
    marajo_basin = gpd.GeoDataFrame(attrs, geometry=[geom], crs=bacias.crs)

print(marajo_basin.drop(columns="geometry"))
print("Marajó Basin bounds [W, S, E, N]:", marajo_basin.total_bounds)

# ============================================================
# 3) Save the Marajó Basin as an isolated shapefile
# ============================================================
output_shape_dir = Path("marajo_basin_shapefile")
output_shape_dir.mkdir(exist_ok=True)

marajo_shp_path = output_shape_dir / "marajo_basin.shp"

# Remove previous shapefile components with the same basename
for old_file in output_shape_dir.glob("marajo_basin.*"):
    if old_file.is_file():
        old_file.unlink()

marajo_basin.to_file(
    marajo_shp_path,
    driver="ESRI Shapefile",
    index=False,
    encoding="UTF-8"
)

print(f"Isolated Marajó Basin shapefile saved to: {marajo_shp_path.resolve()}")

In [ ]:
# Geographic limits for the Marajó Basin regional map
# These limits encompass the basin and provide a small regional margin.
lon_min, lon_max = -52.0, -46.0
lat_min, lat_max = -6.5, 2.0

# Output figure
output_name = "fig_location_marajo.png"

In [ ]:
fig = plt.figure(figsize=(9, 9))
ax = plt.axes(projection=ccrs.PlateCarree())

# Map extent
ax.set_extent(
    [lon_min, lon_max, lat_min, lat_max],
    crs=ccrs.PlateCarree()
)

# Background
ax.set_facecolor("#b7d7ef")

ax.add_feature(
    cfeature.LAND,
    facecolor="#f2efe6",
    edgecolor="none",
    zorder=0
)

ax.add_feature(
    cfeature.OCEAN,
    facecolor="#b7d7ef",
    zorder=0
)

# Borders, coastline and rivers
ax.add_feature(
    cfeature.COASTLINE,
    edgecolor="black",
    linewidth=0.6,
    zorder=1
)

ax.add_feature(
    cfeature.BORDERS,
    edgecolor="black",
    linewidth=0.6,
    zorder=1
)

ax.add_feature(
    cfeature.RIVERS,
    edgecolor="#4bb3fd",
    linewidth=0.9,
    zorder=2
)

# Geographic grid
# Integer-degree ticks inside the selected limits
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.4,
    color="gray",
    alpha=0.6,
    linestyle="--"
)

lon_ticks = np.arange(np.ceil(lon_min), np.floor(lon_max) + 1, 1)
lat_ticks = np.arange(np.ceil(lat_min), np.floor(lat_max) + 1, 1)

gl.xlocator = mticker.FixedLocator(lon_ticks)
gl.ylocator = mticker.FixedLocator(lat_ticks)
gl.xformatter = LongitudeFormatter()
gl.yformatter = LatitudeFormatter()

gl.top_labels = True
gl.bottom_labels = False
gl.left_labels = True
gl.right_labels = False

gl.xlabel_style = {"size": 10}
gl.ylabel_style = {"size": 10}

# Scale bar
scale_lon_start = lon_min + 0.45
scale_lat = lat_min + 0.55
scale_km = 250

# Approximate conversion of longitude degrees to kilometres at mean latitude
mean_lat = (lat_min + lat_max) / 2.0
scale_deg = scale_km / (111.32 * np.cos(np.deg2rad(mean_lat)))

ax.plot(
    [scale_lon_start, scale_lon_start + scale_deg],
    [scale_lat, scale_lat],
    color="black",
    linewidth=3,
    transform=ccrs.PlateCarree(),
    zorder=10
)

ax.plot(
    [scale_lon_start, scale_lon_start],
    [scale_lat - 0.07, scale_lat + 0.07],
    color="black",
    linewidth=2,
    transform=ccrs.PlateCarree(),
    zorder=10
)

ax.plot(
    [scale_lon_start + scale_deg, scale_lon_start + scale_deg],
    [scale_lat - 0.07, scale_lat + 0.07],
    color="black",
    linewidth=2,
    transform=ccrs.PlateCarree(),
    zorder=10
)

ax.text(
    scale_lon_start + scale_deg / 2.0,
    scale_lat + 0.16,
    "250 km",
    ha="center",
    va="bottom",
    fontsize=11,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8),
    transform=ccrs.PlateCarree(),
    zorder=11
)

ax.text(
    lon_min + 0.45,
    lat_min + 0.12,
    "Georeferenced to WGS84",
    ha="left",
    va="bottom",
    fontsize=9,
    bbox=dict(facecolor="white", edgecolor="gray", alpha=0.85),
    transform=ccrs.PlateCarree(),
    zorder=11
)

# ONLY the Marajó Basin sedimentary-basin boundary
marajo_basin.boundary.plot(
    ax=ax,
    color="red",
    linewidth=2.6,
    zorder=6
)

legend_handles = [Line2D([0], [0], color="red", linewidth=2.6, label="Marajó Basin")]
ax.legend(handles=legend_handles,
    loc="upper right",
    frameon=True,
    framealpha=0.95,
    facecolor="white",
    edgecolor="gray",
    fontsize=10)

# Optional title
# ax.set_title("Location of the Marajó Basin study area, northern Brazil", fontsize=13, pad=15)

plt.savefig("results/fig_location_marajo.png", dpi=300, bbox_inches="tight")
plt.show()